In [1]:
import numpy as np
import pandas as pd

# Phase 4 - Historical Simulation VaR & Expected Shortfall

In [2]:
# Step 1: Loading Phase 2 outputs
print('Loading finalizzed portfolio data ...')
portfolio_dollar_value = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\portfolio_dollar_value.csv', index_col =0, parse_dates =True).iloc[:,0]
portfolio_pnl = pd.read_csv(r'C:\Users\sentr\Downloads\Internships\QFI\CSV Files\portfolio_pnl.csv', index_col =0, parse_dates =True).iloc[:,0]

Loading finalizzed portfolio data ...


In [3]:
# Simple return for correct linear dollar scaling
port_simple_returns = portfolio_dollar_value.pct_change().dropna()
print(f'Total available trading days {len(port_simple_returns)}')

Total available trading days 1506


In [4]:
# Step 2: Blueprint parameter and core risk function
windows = [250, 500, 750]
confidence_levels = [0.95, 0.99, 0.995]
def calculate_historical_var_es(returns_window, conf_levels):
    percentile = (1 - conf_levels)*100
    var_pct = -np.percentile(returns_window, percentile, method = 'lower')
    tail_returns = returns_window[returns_window <=-var_pct]
    es_pct = -tail_returns.mean() if len(tail_returns) > 0 else var_pct
    return var_pct, es_pct

In [5]:
# Step 3: Initializing the Rolling Engine
risk_metrics = pd.DataFrame(index = port_simple_returns.index)
risk_metrics['Portfolio_Dollar_Value'] = portfolio_dollar_value.loc[port_simple_returns.index]
risk_metrics['Daily_PnL'] = portfolio_pnl.loc[port_simple_returns.index]
risk_metrics['Portfolio_Dollar_Value_Prior'] = risk_metrics['Portfolio_Dollar_Value'].shift(1)

ret_array = port_simple_returns.values


In [9]:
# Step 4: Executing Rolling Calculations
for w in windows:
    for cl in confidence_levels:
        print(f'Calculating {w} day window at {cl*100}% confidence....')

        var_list = np.full(len(port_simple_returns), np.nan)
        es_list = np.full(len(port_simple_returns), np.nan)

        # Sliding Window: Day ith risk uses only i-w through i-1 days. No look-ahead bias
        for i in range(w, len(ret_array)):
            window_data = ret_array[i-w: i]
            v, e = calculate_historical_var_es(window_data, cl)
            var_list[i] = v
            es_list[i] = e

        col_base = f'{w}d_{cl*100}%'
        risk_metrics[f'VaR_pct_{col_base}'] = var_list
        risk_metrics [f'ES_pct_{col_base}'] = es_list

        # Dollar scaling of VaR
        risk_metrics[f'VaR_1d_$_' + col_base] = risk_metrics[f'VaR_pct_{col_base}'] * risk_metrics['Portfolio_Dollar_Value_Prior']
        risk_metrics[f'ES_1d_$_' + col_base] = risk_metrics[f'ES_pct_{col_base}'] * risk_metrics['Portfolio_Dollar_Value_Prior']

        # 10 day horizon via square root of time scalling.
        risk_metrics[f'VaR_10d_$_' + col_base] = risk_metrics[f'VaR_1d_$' + col_base] *np.sqrt(10)
        risk_metrics[f'ES_10d_$_' + col_base] = risk_metrics[f'ES_1d_$' + col_base] *np.sqrt(10)  


Calculating 250 day window at 95.0% confidence....
Calculating 250 day window at 99.0% confidence....
Calculating 250 day window at 99.5% confidence....
Calculating 500 day window at 95.0% confidence....
Calculating 500 day window at 99.0% confidence....
Calculating 500 day window at 99.5% confidence....
Calculating 750 day window at 95.0% confidence....
Calculating 750 day window at 99.0% confidence....
Calculating 750 day window at 99.5% confidence....


In [11]:
# Step 5: Finalizing the out-of-sample dataset
# The 750-day window is the last column to stop being NaN, so checking it alone is equivalent to a full blanket dropna() 
out_of_sample_risk = risk_metrics.dropna(subset = ['VaR_1d_$_750d_99.0%'])
print(f'Phase 4 complete. Out of sample backtesting days available {len(out_of_sample_risk)}')

# Defensive Check: Confirming nothing else slipped unexpectedly
stray_nan_count = out_of_sample_risk.isna().sum().sum()

if stray_nan_count > 0:
    print(f'{stray_nan_count} unexpected NaN value(s) remain outside the checked column.')
else:
    print('No stray NaN values found elsewhere in the out-of-sample dataset.')


Phase 4 complete. Out of sample backtesting days available 756
No stray NaN values found elsewhere in the out-of-sample dataset.


In [12]:
# Step 6: Blueprint validation
var_1d = out_of_sample_risk['VaR_1d_$_750d_99.0%']
var_10d = out_of_sample_risk['VaR_10d_$_750d_99.0%']
es_1d = out_of_sample_risk['ES_1d_$_750d_99.0%']
 
assert (var_1d > 0).all(), 'Validation Failed: VaR must be a positive loss number on every day.'
assert (var_10d > var_1d).all(), 'Validation Failed: 10-day VaR must exceed 1-day VaR on every day.'
assert (es_1d >= var_1d).all(), 'Validation Failed: ES must be >= VaR on every day.'
 
# The more confidence level gives larger VaR check 
var_95 = out_of_sample_risk['VaR_1d_$_750d_95.0%']
var_99 = out_of_sample_risk['VaR_1d_$_750d_99.0%']
var_995 = out_of_sample_risk['VaR_1d_$_750d_99.5%']
 
assert (var_99 >= var_95).all(), 'Validation Failed: 99% VaR must be >= 95% VaR on every day.'
assert (var_995 >= var_99).all(), 'Validation Failed: 99.5% VaR must be >= 99% VaR on every day.'
 
print('All mathematical constraints validated across the full out-of-sample period.')
 

All mathematical constraints validated across the full out-of-sample period.


In [14]:
# Step 7: Saving results for dashboard and backtesting.
out_of_sample_risk.to_csv('historical_simulation_var_es.csv')
print('Saved: historical_simulation_var_es.csv')

Saved: historical_simulation_var_es.csv
